# Forecast Using the T0 Family


TimeCopilot ships a single `T0` class for The Forecasting Company open-weights checkpoints. Pass the Hugging Face `repo_id` to select the model:

| Release | `repo_id` | Size |
|---------|-----------|------|
| **T0 alpha** (default) | `theforecastingcompany/t0-alpha` | ~102M |
| **T0 beta** | `theforecastingcompany/t0-beta` | ~256M |

*Requirements*:

    - Python 3.11 – 3.13
    - `foundationforecast>=0.1.8` (pulls `tfc-t0>=0.5.0` for beta)
    - CUDA recommended (CPU works but is slower)

**Quantiles:** `t0-alpha` uses 5 native quantile knots; `t0-beta` uses 21 levels (0.01–0.99). Pass `level` or `quantiles`; requested levels are interpolated from native knots.



## Import libraries


In [ ]:
import sys

import pandas as pd

from timecopilot import TimeCopilotForecaster

if sys.version_info < (3, 11) or sys.version_info >= (3, 14):
    raise RuntimeError("T0 requires Python >= 3.11 and < 3.14")


## Load the dataset

The DataFrame must include at least the following columns:
- unique_id: Unique identifier for each time series (string)
- ds: Date column (datetime format)
- y: Target variable for forecasting (float format)


In [ ]:
df = pd.read_csv(
    "https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv",
    parse_dates=["ds"],
)
df.head()


## Plot the data


In [ ]:
TimeCopilotForecaster.plot(df)


## Create a TimeCopilotForecaster

We compare T0 alpha and beta. Each model gets a distinct `alias` so forecasts are easy to identify.


In [ ]:
from timecopilot.models.foundation.t0 import T0

context_length = 512
batch_size = 8

models = [
    T0(
        repo_id="theforecastingcompany/t0-alpha",
        alias="t0-alpha",
        context_length=context_length,
        batch_size=batch_size,
    ),
    T0(
        repo_id="theforecastingcompany/t0-beta",
        alias="t0-beta",
        context_length=context_length,
        batch_size=batch_size,
    ),
]

tcf = TimeCopilotForecaster(models=models)


## Cross-validate with prediction intervals

Beta supports fine-grained quantiles; alpha interpolates from its 5 native knots.


In [ ]:
level = [80, 90]
h = 12
cv_df = tcf.cross_validation(df=df, h=h, level=level)
cv_df.head()


## Plot forecast comparison

One panel per event series. Each panel overlays T0 alpha and beta; **t0-beta** is highlighted with its 80% prediction interval. Saved to `t0-family-forecast.png`.


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.style.use("dark_background")

series_ids = sorted(df["unique_id"].unique())
highlight = "t0-beta"
forecast_models = ["t0-beta", "t0-alpha"]
model_colors = {
    "t0-beta": "#5bc0de",
    "t0-alpha": "#90ee90",
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.ravel()

for ax, series_id in zip(axes, series_ids, strict=True):
    fcst_all = cv_df.query("unique_id == @series_id").sort_values("ds")
    cutoff = fcst_all["cutoff"].iloc[0]
    forecast_start = fcst_all["ds"].min()
    history_start = forecast_start - pd.DateOffset(months=3 * h)

    series_df = df.query("unique_id == @series_id and ds >= @history_start").sort_values("ds")
    fcst_df = (
        cv_df.query("unique_id == @series_id and ds >= @history_start")
        .sort_values("ds")
        .copy()
    )
    insample = series_df.query("ds <= @cutoff")
    forecast = fcst_df.query("ds >= @forecast_start")

    ax.plot(
        insample["ds"],
        insample["y"],
        color="white",
        linewidth=2.0,
        label="Actual",
        zorder=5,
    )
    for model in forecast_models:
        ax.plot(
            forecast["ds"],
            forecast[model],
            color=model_colors[model],
            linewidth=1.8,
            marker="o",
            markersize=3,
            label=model,
            zorder=4,
        )
    ax.fill_between(
        forecast["ds"],
        forecast[f"{highlight}-lo-80"],
        forecast[f"{highlight}-hi-80"],
        color=model_colors[highlight],
        alpha=0.25,
        label=f"{highlight} 80% interval",
        zorder=3,
    )
    ax.axvline(forecast_start, color="0.65", linestyle="--", linewidth=1, zorder=2)
    ax.set_title(series_id, fontsize=13, pad=8)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(loc="upper left", frameon=False, fontsize=7)

fig.suptitle(
    f"Event page-views — T0 family ({h}-month horizon)",
    fontsize=15,
    y=0.98,
)
fig.supxlabel("Date")
fig.supylabel("Page Views")
fig.tight_layout()
fig.savefig(
    "t0-family-forecast.png",
    dpi=150,
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
)
plt.show()


## Evaluate


In [ ]:
from functools import partial

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase, scaled_crps

metrics_df = evaluate(
    cv_df.drop(columns=["cutoff"]),
    train_df=df.query("ds <= '2024-08-31'"),
    metrics=[partial(mase, seasonality=12), scaled_crps],
    level=level,
)
metrics_df.groupby("metric").mean(numeric_only=True).T.sort_values(
    by="scaled_crps"
).round(3)
